# Adding missing columns and frequency checkups

## Adding BMI column 
$$Coverage: 10,6\% (n=150.636)$$
$$Average= 26,96 \pm 6,72$$

In [6]:
import pandas as pd
import numpy as np

# 1. Carregar el fitxer COMORBIDITIES
csv_path = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\2_Comorbidities\COMORBIDITIES.csv'
dfc = pd.read_csv(csv_path)
n_files = len(dfc)

# 2. Generar valors seguint una distribució normal (mitjana=26.96, SD=6.72)
# Utilitzem np.random.normal per simular la variabilitat real
bmi_generat = np.random.normal(loc=26.96, scale=6.72, size=n_files)

# 3. Aplicar els límits de l'estudi (excloure < 10 i > 65)
bmi_generat = np.clip(bmi_generat, 10, 65)

# 4. Crear la cobertura del 10.6% (el 89.4% seran NaN)
# Creem un filtre on 1 significa "té dada" i 0 significa "buit"
cobertura = np.random.choice([1, 0], size=n_files, p=[0.106, 0.894])

# Apliquem el filtre
bmi_final = np.where(cobertura == 1, bmi_generat, np.nan)

# 5. Afegir la columna i guardar
dfc['bmi'] = bmi_final
dfc.to_csv(csv_path, index=False)

# 6. Verificació de les estadístiques
print(f"Cobertura real: {(dfc['bmi'].notnull().sum() / n_files * 100):.2f}%")
print(f"Mitjana: {dfc['bmi'].mean():.2f}")
print(f"Desviació Estàndard (SD): {dfc['bmi'].std():.2f}")

Cobertura real: 10.60%
Mitjana: 27.80
Desviació Estàndard (SD): 7.55


## Comorbidities count
Comorbidities: 0, 1 o 2 (52.4% (0), 22.5% (1) y 25.1% (2)) 

In [8]:
dfc.comorbidities.value_counts(normalize=True)*100

comorbidities
0    52.4
2    25.1
1    22.5
Name: proportion, dtype: float64

In [11]:
df_long = pd.melt(dfc, 
                  id_vars=['id'], 
                  value_vars=['comorbidities_esp_1', 'comorbidities_esp_2'],
                  value_name='comorbidity')

# Neteja: Eliminem les files on no hi havia cap malaltia (NaN) i la columna 'variable' que crea el melt automàticament
df_long = df_long.dropna(subset=['comorbidity']).drop(columns='variable')
df_long = df_long[df_long['comorbidity'] != 0]
df_long = df_long[df_long['comorbidity'] != '0']
# Ordenar per ID de pacient per a que quedi clar
df_long = df_long.sort_values(by='id').reset_index(drop=True)
# Guardar el resultat en un fitxer nou
#df_long.to_csv('COMORBIDITIES_LONG.csv', index=False)
df_long = df_long.sort_values(by='id').reset_index(drop=True)
df_long.to_csv('COMORBIDITIES_LONG.csv', index=False)

Checkup comorbidities count:

Number of medical comorbidities: (Prevalencia de enfermedades específicas en el total de la muestra, N = 1,403,566): 

'Musculoskeletal Disorders': N = 601,843 (42.88%). 

'Hypertension': N = 261,633 (18.64%). 

'Arthrosis': N = 185,634 (13.23%). 

'Asthma': N = 125,401 (8.93%). 

'Diabetes': N = 103,281 (7.36%). 

'Neoplasms': N = 91,566 (6.52%). 

'Arthritis': N = 90,673 (6.46%). 

'COPD': N = 69,378 (4.94%). 

'Osteoporosis': N = 61,346 (4.37%). 

'Chronic kidney disease': N = 46,747 (3.33%). 

'Ischemic stroke': N = 38,569 (2.75%). 

'Heart failure': N = 33,696 (2.40%). 

'Dementia': N = 32,712 (2.33%). 

'Cirrhosis': N = 7,505 (0.53%). 

In [15]:
df_long.comorbidity.value_counts(normalize=True, dropna=True)*100

comorbidity
Musculoskeletal Disorders    14.442916
Arthrosis                    11.966988
Hypertension                 10.178817
Asthma                        8.115543
Osteoporosis                  7.427785
Ischemic stroke               7.152682
Diabetes                      7.015131
COPD                          7.015131
Chronic kidney disease        6.877579
Dementia                      5.639615
Neoplasms                     5.089409
Arthritis                     4.951857
Heart failure                 4.126547
Name: proportion, dtype: float64

In [ ]:
import pandas as pd
import numpy as np

# 1. Carregar el fitxer original
ruta_comorb = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\2_Comorbidities\GMA.csv'
dfc = pd.read_csv(ruta_comorb)
n = len(dfc)

# 2. Definim les prevalences REALS de l'estudi PADRIS-PRESTO
prevalences = {
    'Musculoskeletal Disorders': 0.4288,
    'Hypertension': 0.1864,
    'Arthrosis': 0.1323,
    'Asthma': 0.0893,
    'Diabetes': 0.0736,
    'Neoplasms': 0.0652,
    'Arthritis': 0.0646,
    'COPD': 0.0494,
    'Osteoporosis': 0.0437,
    'Chronic kidney disease': 0.0333,
    'Ischemic stroke': 0.0275,
    'Heart failure': 0.0240,
    'Dementia': 0.0233,
    'Cirrhosis': 0.0053,
    'AIDS': 0.0051
}

# 3. Generem cada malaltia de forma independent
# Crearem una llista on guardarem les malalties de cada pacient
pacients_comorb = [[] for _ in range(n)]

for malaltia, prob in prevalences.items():
    # Per a cada malaltia, fem un sorteig de Sí (1) o No (0) per a TOTS els pacients
    resultats = np.random.binomial(1, prob, n)
    for i, positiu in enumerate(resultats):
        if positiu == 1:
            pacients_comorb[i].append(malaltia)

# 4. Omplim les columnes esp_1 i esp_2 amb el resultat
# Si un pacient té més de 2, agafem les dues primeres. Si en té 1, la segona és 0.
dfc['comorbidities_esp_1'] = [p[0] if len(p) > 0 else '0' for p in pacients_comorb]
dfc['comorbidities_esp_2'] = [p[1] if len(p) > 1 else '0' for p in pacients_comorb]

# 5. Guardar i Verificar
dfc.to_csv(ruta_comorb, index=False)

# Verificació ràpida (Recompte de la columna 1 per veure si ha pujat)
print("--- Nova Prevalença a la Columna 1 ---")
print((df_c['comorbidities_esp_1'].value_counts(normalize=True) * 100).round(2))

--- Prevalença segons la població total (estil PADRIS-PRESTO) ---
comorbidity
Musculoskeletal Disorders    10.5%
Arthrosis                     8.7%
Hypertension                  7.4%
Asthma                        5.9%
Osteoporosis                  5.4%
Ischemic stroke               5.2%
Diabetes                      5.1%
COPD                          5.1%
Chronic kidney disease        5.0%
Dementia                      4.1%
Neoplasms                     3.7%
Arthritis                     3.6%
Heart failure                 3.0%
Name: count, dtype: str
